In [2]:
'''imports'''
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, accuracy_score, roc_auc_score, precision_score,
                             recall_score, f1_score, confusion_matrix,
                             precision_recall_curve)


import keras
from keras.models import Sequential
from keras.layers import Input, Dense, Dropout
from keras.callbacks import EarlyStopping, ModelCheckpoint

from preprocessing import prepare_data


from sklearn.neural_network import MLPClassifier
from imblearn.over_sampling import SMOTE





In [14]:
'''Prepare_data, val split, scaling, smote'''
# 1. Prepare data (Season is already one-hot encoded; missing CHL imputed on train_val)
X_train_val, y_train_val, X_test, y_test = prepare_data(
    strategy="doy_window_median",
    split_test="stratified",
    one_hot_cluster=True,
    sin_doy=True,
    scale=False,
)

# 2. Validation split from train_val only (test stays untouched)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.15,
    random_state=42,
    stratify=y_train_val,
)

# 3. Scale continuous features only (fit on X_train)
cont_cols = ["SST", "UI", "YDOY", "sin_doy", "CHL_log", "CHL_uncertainty"]
scaler = StandardScaler()

X_train_sc = X_train.copy()
X_val_sc = X_val.copy()
X_test_sc = X_test.copy()

X_train_sc.loc[:, cont_cols] = scaler.fit_transform(X_train[cont_cols])
X_val_sc.loc[:, cont_cols] = scaler.transform(X_val[cont_cols])
X_test_sc.loc[:, cont_cols] = scaler.transform(X_test[cont_cols])

# 4. Apply SMOTE ONLY to the scaled training set
smote = SMOTE(sampling_strategy="auto", k_neighbors=5, random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train_sc, y_train)

print("Before SMOTE:", y_train.value_counts().to_dict())
print("After SMOTE: ", pd.Series(y_train_sm).value_counts().to_dict())


Before SMOTE: {0: 673, 1: 94}
After SMOTE:  {0: 673, 1: 673}


In [5]:
'''Way 1'''
# 5. Train Neural Network (use X_val_sc to tune architecture/threshold/early stopping)
nn = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation="relu",
    alpha=1e-3,
    max_iter=500,
    early_stopping=True,
    validation_fraction=0.15,
    random_state=42,
)
nn.fit(X_train_sm, y_train_sm)


'''Prints'''

val_proba = nn.predict_proba(X_val_sc)[:, 1]
print("\n--- Validation ---")
print("ROC-AUC:", round(roc_auc_score(y_val, val_proba), 4))
print(classification_report(y_val, (val_proba >= 0.5).astype(int), digits=3))

# Final check on untouched test set
test_proba = nn.predict_proba(X_test_sc)[:, 1]
print("--- Test ---")
print("ROC-AUC:", round(roc_auc_score(y_test, test_proba), 4))
print(classification_report(y_test, (test_proba >= 0.5).astype(int), digits=3))



--- Validation ---
ROC-AUC: 0.784
              precision    recall  f1-score   support

           0      0.950     0.798     0.868       119
           1      0.333     0.706     0.453        17

    accuracy                          0.787       136
   macro avg      0.642     0.752     0.660       136
weighted avg      0.873     0.787     0.816       136

--- Test ---
ROC-AUC: 0.7962
              precision    recall  f1-score   support

           0      0.914     0.722     0.807       133
           1      0.327     0.667     0.439        27

    accuracy                          0.713       160
   macro avg      0.621     0.694     0.623       160
weighted avg      0.815     0.713     0.745       160



In [15]:
'''Way 2'''
n_features = X_train_sm.shape[1]
BATCH_SIZE = 32
LR = 1e-3

# Convert to clean numpy arrays (keep validation and test untouched)
X_tr = np.asarray(X_train_sm, dtype=np.float32)
y_tr = np.asarray(y_train_sm, dtype=np.float32)
X_va = np.asarray(X_val_sc, dtype=np.float32)
y_va = np.asarray(y_val, dtype=np.float32)
X_te = np.asarray(X_test_sc, dtype=np.float32)
y_te = np.asarray(y_test, dtype=np.float32)


def build_mlp(units, dropout=0.0, name="mlp", regul=True, reg=1e-4, lr=LR, seed=42):
    keras.utils.set_random_seed(seed)
    m = Sequential(name=name)
    m.add(Input(shape=(n_features,)))
    for u in units:
        reg_fn = keras.regularizers.l2(reg) if regul else None
        m.add(Dense(u, activation="relu", kernel_regularizer=reg_fn))
        if dropout > 0:
            m.add(Dropout(dropout))
    m.add(Dense(1, activation="sigmoid"))
    m.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss="binary_crossentropy",
        metrics=["accuracy", keras.metrics.AUC(name="auc")],
    )
    return m


def train_mlp(model, name, n_epochs, pat, batch_size=BATCH_SIZE, class_weight=None):
    cbs = [
        EarlyStopping(
            monitor="val_auc",
            mode="max",
            patience=pat,
            start_from_epoch=20,
            restore_best_weights=True,
        ),
        ModelCheckpoint(
            f"best_{name}.keras",
            monitor="val_auc",
            mode="max",
            save_best_only=True,
        ),
    ]
    return model.fit(
        X_tr,
        y_tr,
        epochs=n_epochs,
        batch_size=batch_size,
        validation_data=(X_va, y_va),
        callbacks=cbs,
        class_weight=class_weight,  # None when already using SMOTE
        verbose=1,
    )


model_A = build_mlp([16, 8], dropout=0.0, name="A_small", regul=True, reg=1e-3)
model_B = build_mlp([64, 32], dropout=0.0, name="B_big_dropout", regul=True, reg=1e-3)

hist_A = train_mlp(model_A, "A_small", n_epochs=300, pat=10, class_weight=None)
hist_B = train_mlp(model_B, "B_big_dropout", n_epochs=400, pat=10, class_weight=None)


'''Prints'''

THRESHOLD = None 

def report(model, name):
    ytr = np.asarray(y_train_sm, dtype=int)
    yva = np.asarray(y_val, dtype=int)
    yte = np.asarray(y_test, dtype=int)

    p_tr = model.predict(X_tr, verbose=0).ravel()
    p_va = model.predict(X_va, verbose=0).ravel()
    p_te = model.predict(X_te, verbose=0).ravel()

    if THRESHOLD is None:
        prec, rec, thr = precision_recall_curve(yva, p_va)
        f1 = 2 * prec * rec / (prec + rec + 1e-12)
        t = float(thr[np.argmax(f1[:-1])])
    else:
        t = THRESHOLD

    pred = lambda p: (p >= t).astype(int)

    print(f"===== {name} =====")
    print(f"Train       accuracy = {accuracy_score(ytr, pred(p_tr)):.4f} | AUC = {roc_auc_score(ytr, p_tr):.4f}")
    print(f"Validation  accuracy = {accuracy_score(yva, pred(p_va)):.4f} | AUC = {roc_auc_score(yva, p_va):.4f}")
    print(f"Test        accuracy = {accuracy_score(yte, pred(p_te)):.4f} | AUC = {roc_auc_score(yte, p_te):.4f}")

    print(f"\nThreshold used: {t:.3f}")

    yp = pred(p_te)
    print(f"\n* accuracy on test : {accuracy_score(yte, yp):.4f}")
    print(f"* precision on test: {precision_score(yte, yp, zero_division=0):.4f}")
    print(f"* recall on test   : {recall_score(yte, yp, zero_division=0):.4f}")
    print(f"* f1-score on test : {f1_score(yte, yp, zero_division=0):.4f}")

    print("\nConfusion matrix (rows = true 0/1, cols = predicted 0/1):")
    print(confusion_matrix(yte, yp))
    print()

report(model_A, "A_small")
report(model_B, "B_big_dropout")




'''

THRESHOLD = None   # None = choose automatically on validation (max F1); or set a number, e.g. 0.25

def report(model, name):
    ytr, yva, yte = y_train.astype(int), y_val.astype(int), y_test.astype(int)
    p_tr = model.predict(X_train, verbose=0).ravel()
    p_va = model.predict(X_val,   verbose=0).ravel()
    p_te = model.predict(X_test,  verbose=0).ravel()

    if THRESHOLD is None:
        prec, rec, thr = precision_recall_curve(yva, p_va)
        f1 = 2 * prec * rec / (prec + rec + 1e-12)
        t = float(thr[np.argmax(f1[:-1])])
    else:
        t = THRESHOLD

    pred = lambda p: (p >= t).astype(int)

    print(f"===== {name} =====")
    print(f"Train       accuracy = {accuracy_score(ytr, pred(p_tr)):.4f} | AUC = {roc_auc_score(ytr, p_tr):.4f}")
    print(f"Validation  accuracy = {accuracy_score(yva, pred(p_va)):.4f} | AUC = {roc_auc_score(yva, p_va):.4f}")
    print(f"Test        accuracy = {accuracy_score(yte, pred(p_te)):.4f} | AUC = {roc_auc_score(yte, p_te):.4f}")

    print(f"\nThreshold used: {t:.3f}")

    yp = pred(p_te)
    print(f"\n* accuracy on test : {accuracy_score(yte, yp):.4f}")
    print(f"* precision on test: {precision_score(yte, yp, zero_division=0):.4f}")
    print(f"* recall on test   : {recall_score(yte, yp, zero_division=0):.4f}")
    print(f"* f1-score on test : {f1_score(yte, yp, zero_division=0):.4f}")

    print("\nConfusion matrix (rows = true 0/1, cols = predicted 0/1):")
    print(confusion_matrix(yte, yp))
    print()

report(model_A, "A_small")
report(model_B, "B_big_dropout")

'''

Epoch 1/300
43/43 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.6122 - auc: 0.6460 - loss: 0.6945 - val_accuracy: 0.6618 - val_auc: 0.7301 - val_loss: 0.6298
Epoch 2/300
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7117 - auc: 0.7725 - loss: 0.6333 - val_accuracy: 0.7279 - val_auc: 0.7860 - val_loss: 0.5529
Epoch 3/300
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7392 - auc: 0.8166 - loss: 0.5819 - val_accuracy: 0.7132 - val_auc: 0.7961 - val_loss: 0.5102
Epoch 4/300
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7526 - auc: 0.8347 - loss: 0.5410 - val_accuracy: 0.7132 - val_auc: 0.8028 - val_loss: 0.4947
Epoch 5/300
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7660 - auc: 0.8480 - loss: 0.5105 - val_accuracy: 0.7279 - val_auc: 0.8089 - val_loss: 0.4894
Epoch 6/300
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7756 - auc: 0.8606 - loss: 0.4872 - val_accuracy: 0.7059 - val_auc: 0.8072 - val_loss: 0.4868
Epoch 7/300
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms

'\n\nTHRESHOLD = None   # None = choose automatically on validation (max F1); or set a number, e.g. 0.25\n\ndef report(model, name):\n    ytr, yva, yte = y_train.astype(int), y_val.astype(int), y_test.astype(int)\n    p_tr = model.predict(X_train, verbose=0).ravel()\n    p_va = model.predict(X_val,   verbose=0).ravel()\n    p_te = model.predict(X_test,  verbose=0).ravel()\n\n    if THRESHOLD is None:\n        prec, rec, thr = precision_recall_curve(yva, p_va)\n        f1 = 2 * prec * rec / (prec + rec + 1e-12)\n        t = float(thr[np.argmax(f1[:-1])])\n    else:\n        t = THRESHOLD\n\n    pred = lambda p: (p >= t).astype(int)\n\n    print(f"===== {name} =====")\n    print(f"Train       accuracy = {accuracy_score(ytr, pred(p_tr)):.4f} | AUC = {roc_auc_score(ytr, p_tr):.4f}")\n    print(f"Validation  accuracy = {accuracy_score(yva, pred(p_va)):.4f} | AUC = {roc_auc_score(yva, p_va):.4f}")\n    print(f"Test        accuracy = {accuracy_score(yte, pred(p_te)):.4f} | AUC = {roc_auc_scor